# 00 — Structured Data — All in One
Exécute ce notebook si tu veux refaire tout le pipeline dans un seul fichier.

**Avant exécution : modifie `src/config.py`.**


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from config import DATA_DIR, OUTPUT_DIR, FILES, TARGET_RMPM_IDS
from structured_utils import *

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)


## 1. Chargement


In [ ]:
datasets = {}
for key, filename in FILES.items():
    path = DATA_DIR / filename
    print(f"{key}: {path} | exists={path.exists()}")
    datasets[key] = load_excel(path)
    print("shape:", datasets[key].shape)


In [ ]:
for name, df in datasets.items():
    print("\n" + "="*100)
    print(name.upper(), df.shape)
    print("="*100)
    display(df.head())
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_pct": (df.isna().mean()*100).round(2)
    }).sort_values("missing_pct", ascending=False))
    print("duplicates:", df.duplicated().sum())
    if "RMPM_ID" in df.columns:
        print("RMPM_ID uniques:", df["RMPM_ID"].nunique())


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in datasets.items():
    df.to_pickle(OUTPUT_DIR / f"00_raw_{name}.pkl")
print("Raw snapshots saved.")


## 2. Nettoyage / préparation


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR / "00_raw_rmpm.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR / "00_raw_legal_entity.pkl")
facility = pd.read_pickle(OUTPUT_DIR / "00_raw_facility.pkl")
covenant = pd.read_pickle(OUTPUT_DIR / "00_raw_covenant.pkl")
asset = pd.read_pickle(OUTPUT_DIR / "00_raw_asset.pkl")


In [ ]:
for df in [rmpm, legal_entity, facility, covenant, asset]:
    if "RMPM_ID" in df.columns:
        df["RMPM_ID"] = clean_identifier(df["RMPM_ID"])
    if "FACILITY_UNIQUE_ID" in df.columns:
        df["FACILITY_UNIQUE_ID"] = clean_identifier(df["FACILITY_UNIQUE_ID"])

rmpm = filter_target_counterparties(rmpm, TARGET_RMPM_IDS)
legal_entity = filter_target_counterparties(legal_entity, TARGET_RMPM_IDS)
facility = filter_target_counterparties(facility, TARGET_RMPM_IDS)
covenant = filter_target_counterparties(covenant, TARGET_RMPM_IDS)
asset = filter_target_counterparties(asset, TARGET_RMPM_IDS)

rmpm = convert_date_columns(rmpm)
legal_entity = convert_date_columns(legal_entity)
facility = convert_date_columns(facility)
covenant = convert_date_columns(covenant)
asset = convert_date_columns(asset)

legal_entity = normalize_boolean_columns(legal_entity)
facility = normalize_boolean_columns(facility)
covenant = normalize_boolean_columns(covenant)


In [ ]:
for c in ["PD_CLEAN", "GRR_SU"]:
    if c in rmpm.columns:
        rmpm[c] = rmpm[c].map(parse_number)

# Valeurs invalides documentees : PD = 999.99 (sentinelle), negatives,
# et PD superieures a 100 quelle que soit l'unite (fraction ou pourcentage).
# On ne convertit PAS automatiquement entre fraction et pourcentage :
# verifier la convention metier avant toute interpretation.
if "PD_CLEAN" in rmpm.columns:
    invalid_pd = (rmpm["PD_CLEAN"].eq(999.99)
                  | rmpm["PD_CLEAN"].lt(0)
                  | rmpm["PD_CLEAN"].gt(100)).fillna(False)
    rmpm["PD_ANOMALY"] = invalid_pd
    print("Observations PD invalides exclues des courbes :", int(invalid_pd.sum()))
    rmpm.loc[invalid_pd, "PD_CLEAN"] = np.nan

for c in ["FINAL_TAKE_AMOUNT", "OUTSTANDING_GLOBAL", "NET_AMOUNT", "GROSS_AMOUNT", "TENOR", "TENOR_YEARS"]:
    if c in facility.columns:
        facility[c] = facility[c].map(parse_number)

for c in ["SAF", "TOTAL_SAF"]:
    if c in legal_entity.columns:
        legal_entity[c] = legal_entity[c].map(parse_number)

In [ ]:
if "SITUATION_DATE" in rmpm.columns:
    rmpm = rmpm.sort_values(["RMPM_ID","SITUATION_DATE"]).reset_index(drop=True)
if "DATE_OF_COMMITTEE" in legal_entity.columns:
    legal_entity = legal_entity.sort_values(["RMPM_ID","DATE_OF_COMMITTEE"]).reset_index(drop=True)
facility_sort = [c for c in ["RMPM_ID","FACILITY_UNIQUE_ID","DATE_OF_COMMITTEE"] if c in facility.columns]
if facility_sort:
    facility = facility.sort_values(facility_sort).reset_index(drop=True)

kyc_cols = [c for c in legal_entity.columns if "KYC" in canonical_column_name(c)]
if kyc_cols:
    legal_entity[kyc_cols] = legal_entity.groupby("RMPM_ID", group_keys=False)[kyc_cols].ffill()

rmpm_rating_col = find_column(rmpm, "Counterparty Rating", "COUNTERPARTY_RATING")
if rmpm_rating_col:
    rmpm["RATING_NUM"] = rmpm[rmpm_rating_col].map(parse_rating)
legal_rating_col = find_column(legal_entity, "COUNTERPARTY_RATING", "Counterparty Rating")
if legal_rating_col:
    legal_entity["RATING_NUM"] = legal_entity[legal_rating_col].map(parse_rating)


In [ ]:
for name, df in {
    "rmpm":rmpm, "legal_entity":legal_entity, "facility":facility,
    "covenant":covenant, "asset":asset
}.items():
    df.to_pickle(OUTPUT_DIR / f"01_clean_{name}.pkl")
    print(name, df.shape)


## 3. Coverage / qualité


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR / "01_clean_rmpm.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR / "01_clean_legal_entity.pkl")
facility = pd.read_pickle(OUTPUT_DIR / "01_clean_facility.pkl")
covenant = pd.read_pickle(OUTPUT_DIR / "01_clean_covenant.pkl")
asset = pd.read_pickle(OUTPUT_DIR / "01_clean_asset.pkl")


In [ ]:
coverage_df = build_coverage(rmpm, legal_entity, facility, covenant, asset, TARGET_RMPM_IDS)
display(coverage_df)
coverage_df.to_csv(OUTPUT_DIR/"coverage.csv", index=False)


In [ ]:
for name, df in {"RMPM":rmpm,"LEGAL":legal_entity,"FACILITY":facility,"COVENANT":covenant,"ASSET":asset}.items():
    print("\n", name)
    print("rows:", len(df))
    print("duplicates:", df.duplicated().sum())
    print("missing RMPM_ID:", df["RMPM_ID"].isna().sum() if "RMPM_ID" in df.columns else "N/A")


## 4. RMPM


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR / "01_clean_rmpm.pkl")


In [ ]:
for column, delta in [("PD_CLEAN","DELTA_PD"),("GRR_SU","DELTA_GRR"),("RATING_NUM","DELTA_RATING")]:
    if column in rmpm.columns:
        rmpm[delta] = rmpm.groupby("RMPM_ID")[column].diff()
display(rmpm.head(20))


In [ ]:
def plot_metric(cp, metric, ylabel=None):
    data = rmpm[rmpm["RMPM_ID"] == cp].sort_values("SITUATION_DATE")
    if metric not in data.columns:
        print("Missing:", metric); return
    plt.figure(figsize=(14,5))
    plt.plot(data["SITUATION_DATE"], data[metric], marker="o")
    plt.title(f"{metric} trajectory - {cp}")
    plt.xlabel("Date"); plt.ylabel(ylabel or metric); plt.grid(alpha=.3)
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for cp in TARGET_RMPM_IDS:
    plot_metric(cp, "PD_CLEAN", "PD")
    plot_metric(cp, "GRR_SU", "GRR")
    plot_metric(cp, "RATING_NUM", "Rating")


In [ ]:
rmpm.to_pickle(OUTPUT_DIR/"02_rmpm_enriched.pkl")


## 5. Legal Entity


In [ ]:
legal_entity = pd.read_pickle(OUTPUT_DIR / "01_clean_legal_entity.pkl")


In [ ]:
variables = [
    "COUNTERPARTY_RATING","INTRINSIC_RATING","KYC","WATCHLIST_INDICATOR",
    "UTP_TEST","CONFIRMED_UTP","NPE","Close to Default","SAF","TOTAL_SAF"
]
present = [c for c in variables if c in legal_entity.columns]
print("Variables present:", present)
legal_changes = extract_changes(legal_entity, present)
display(legal_changes)
legal_changes.to_csv(OUTPUT_DIR/"legal_entity_changes.csv", index=False)


## 6. Facilities


In [ ]:
facility = pd.read_pickle(OUTPUT_DIR / "01_clean_facility.pkl")


In [ ]:
if "OUTSTANDING_GLOBAL" in facility.columns and "FINAL_TAKE_AMOUNT" in facility.columns:
    facility["UTILIZATION"] = facility["OUTSTANDING_GLOBAL"] / facility["FINAL_TAKE_AMOUNT"]
    facility.loc[facility["FINAL_TAKE_AMOUNT"] == 0, "UTILIZATION"] = np.nan

agg = {"FACILITY_UNIQUE_ID":"nunique"}
if "FINAL_TAKE_AMOUNT" in facility.columns: agg["FINAL_TAKE_AMOUNT"] = "sum"
if "OUTSTANDING_GLOBAL" in facility.columns: agg["OUTSTANDING_GLOBAL"] = "sum"

facility_summary = facility.groupby(["RMPM_ID","DATE_OF_COMMITTEE"]).agg(agg).reset_index()
facility_summary = facility_summary.rename(columns={
    "FACILITY_UNIQUE_ID":"NB_FACILITIES",
    "FINAL_TAKE_AMOUNT":"TOTAL_FINAL_TAKE",
    "OUTSTANDING_GLOBAL":"TOTAL_OUTSTANDING"
})
if "TOTAL_FINAL_TAKE" in facility_summary.columns and "TOTAL_OUTSTANDING" in facility_summary.columns:
    facility_summary["UTILIZATION"] = facility_summary["TOTAL_OUTSTANDING"] / facility_summary["TOTAL_FINAL_TAKE"]
display(facility_summary.head())


In [ ]:
def plot_facility_amounts(cp):
    d = facility_summary[facility_summary["RMPM_ID"]==cp].sort_values("DATE_OF_COMMITTEE")
    plt.figure(figsize=(15,6))
    if "TOTAL_FINAL_TAKE" in d: plt.plot(d["DATE_OF_COMMITTEE"], d["TOTAL_FINAL_TAKE"], marker="o", label="Final Take")
    if "TOTAL_OUTSTANDING" in d: plt.plot(d["DATE_OF_COMMITTEE"], d["TOTAL_OUTSTANDING"], marker="o", label="Outstanding")
    plt.title(f"Facilities exposure - {cp}"); plt.legend(); plt.grid(alpha=.3)
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for cp in TARGET_RMPM_IDS:
    plot_facility_amounts(cp)


In [ ]:
facility_change_df = facility_changes(facility)
display(facility_change_df)
facility.to_pickle(OUTPUT_DIR/"02_facility_enriched.pkl")
facility_summary.to_csv(OUTPUT_DIR/"facility_summary.csv", index=False)
facility_change_df.to_csv(OUTPUT_DIR/"facility_changes.csv", index=False)


## 7. Covenants / Assets


In [ ]:
covenant = pd.read_pickle(OUTPUT_DIR / "01_clean_covenant.pkl")
asset = pd.read_pickle(OUTPUT_DIR / "01_clean_asset.pkl")


In [ ]:
import re

def parse_breach_status(value):
    """Retourne True uniquement pour une rupture explicite, pas 'NOT BREACHED'."""
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        return bool(value == 1)
    text = str(value).strip().lower().replace("_", " ").replace("-", " ")
    if re.search(r"\b(not|no|non|without|never)\s+breach(ed)?\b", text):
        return False
    if any(term in text for term in ("unbreached", "compliant", "waived", "performing", "no default",
                                     "potential", "suspect", "pending", "risk of", "at risk", "expected", "forecast")):
        return False
    if text in {"1", "true", "yes", "y"}:
        return True
    return bool(re.search(r"\bbreach(ed)?\b", text))

status_col = next((c for c in ["BREACH_STATUS", "BREACHED_STATUS"] if c in covenant.columns), None)
if status_col is None:
    status_col = find_column(covenant, "BREACH_STATUS", "BREACHED_STATUS", "STATUS")
type_col = find_column(covenant, "COVENANT_SUR_NAME", "COVENANT_TYPE", "TYPE")

if status_col:
    display(covenant[status_col].value_counts(dropna=False).to_frame("observations"))
    covenant["IS_BREACH"] = covenant[status_col].map(parse_breach_status).astype(bool)
    print("Observations en rupture (pas necessairement des incidents distincts) :",
          int(covenant["IS_BREACH"].sum()))
    display(covenant.loc[covenant["IS_BREACH"]])
else:
    covenant["IS_BREACH"] = False
    print("Aucune colonne de statut de rupture identifiee : verifier le fichier Covenant.")

In [ ]:
date_candidates = [c for c in ["PERFORMED_ON","RECEIPT_DATE","DUE_DATE","REPORTING_DATE","DATE_OF_COMMITTEE"] if c in covenant.columns]
if date_candidates:
    covenant["COVENANT_EVENT_DATE"] = covenant[date_candidates].bfill(axis=1).iloc[:,0]


In [ ]:
display(asset.groupby("RMPM_ID").size().rename("asset_rows").reset_index())
numeric = asset.select_dtypes(include=np.number).columns.tolist()
print("Numeric asset columns:", numeric)
if numeric:
    display(asset.groupby("RMPM_ID")[numeric].describe())
covenant.to_pickle(OUTPUT_DIR/"02_covenant_enriched.pkl")


## 8. Events / Timeline


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR/"02_rmpm_enriched.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR/"01_clean_legal_entity.pkl")
facility_changes_df = pd.read_csv(OUTPUT_DIR/"facility_changes.csv")
covenant = pd.read_pickle(OUTPUT_DIR/"02_covenant_enriched.pkl")


In [ ]:
events = []

for cp, group in rmpm.groupby("RMPM_ID"):
    group = group.sort_values("SITUATION_DATE")
    for col, label in [("PD_CLEAN","PD change"),("GRR_SU","GRR change"),("RATING_NUM","Rating change")]:
        if col not in group.columns: continue
        previous = group[col].shift(1)
        mask = group[col].notna() & previous.notna() & (group[col] != previous)
        for idx in group[mask].index:
            events.append({
                "RMPM_ID":cp,"DATE":group.loc[idx,"SITUATION_DATE"],"SOURCE":"RMPM",
                "EVENT_TYPE":label,"OLD_VALUE":previous.loc[idx],"NEW_VALUE":group.loc[idx,col],
                "FACILITY_UNIQUE_ID":pd.NA
            })


In [ ]:
for col, on, off in [
    (find_column(legal_entity,"WATCHLIST_INDICATOR"),"Watchlist entry","Watchlist exit"),
    (find_column(legal_entity,"UTP_TEST"),"UTP entry","UTP exit"),
    (find_column(legal_entity,"CONFIRMED_UTP"),"Confirmed UTP","Confirmed UTP exit"),
    (find_column(legal_entity,"NPE"),"NPE entry","NPE exit"),
    (find_column(legal_entity,"Close to Default"),"Close to Default","Close to Default exit"),
]:
    if col:
        events += extract_boolean_transitions(legal_entity,col,"DATE_OF_COMMITTEE",on,off)


In [ ]:
if not facility_changes_df.empty:
    for _, row in facility_changes_df.iterrows():
        events.append({
            "RMPM_ID": row["RMPM_ID"], "DATE": row["DATE"], "SOURCE": "FACILITY",
            "EVENT_TYPE": f"Facility - {row['VARIABLE']} change",
            "OLD_VALUE": row["OLD_VALUE"], "NEW_VALUE": row["NEW_VALUE"],
            "FACILITY_UNIQUE_ID": row.get("FACILITY_UNIQUE_ID", pd.NA)
        })

# Un constat BREACHED est compte comme une OBSERVATION de rupture,
# et non automatiquement comme un incident distinct ou un EWS predictif.
if status_col and "COVENANT_EVENT_DATE" in covenant.columns:
    for _, row in covenant.iterrows():
        status = str(row.get(status_col, ""))
        if bool(row.get("IS_BREACH", False)):
            event_type = "Covenant breach observed"
        elif "waiv" in status.lower():
            event_type = "Covenant waiver observed"
        else:
            continue
        events.append({
            "RMPM_ID": row.get("RMPM_ID"), "DATE": row.get("COVENANT_EVENT_DATE"),
            "SOURCE": "COVENANT", "EVENT_TYPE": event_type,
            "OLD_VALUE": pd.NA,
            "NEW_VALUE": row.get(type_col, pd.NA) if type_col else pd.NA,
            "FACILITY_UNIQUE_ID": row.get("FACILITY_UNIQUE_ID", pd.NA)
        })

expected_cols = ["RMPM_ID", "DATE", "SOURCE", "EVENT_TYPE", "OLD_VALUE", "NEW_VALUE", "FACILITY_UNIQUE_ID"]
events = pd.DataFrame(events, columns=expected_cols)
events["DATE"] = pd.to_datetime(events["DATE"], errors="coerce", utc=True).dt.tz_localize(None)
events = events.dropna(subset=["RMPM_ID", "DATE"]).sort_values(["RMPM_ID", "DATE"]).reset_index(drop=True)
display(events)
events.to_csv(OUTPUT_DIR / "events.csv", index=False)
events.to_pickle(OUTPUT_DIR / "events.pkl")

In [ ]:
def plot_event_timeline(cp):
    d = events[events["RMPM_ID"]==cp].copy()
    if d.empty: return
    labels = d["EVENT_TYPE"].dropna().unique().tolist()
    ymap = {e:i for i,e in enumerate(labels)}
    d["Y"] = d["EVENT_TYPE"].map(ymap)
    plt.figure(figsize=(16,max(6,len(labels)*.45)))
    plt.scatter(d["DATE"],d["Y"],s=70)
    plt.yticks(range(len(labels)),labels)
    plt.title(f"Event timeline - {cp}")
    plt.grid(axis="x",alpha=.3); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for cp in TARGET_RMPM_IDS:
    plot_event_timeline(cp)


## 9. Export final


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR/"02_rmpm_enriched.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR/"01_clean_legal_entity.pkl")
facility = pd.read_pickle(OUTPUT_DIR/"02_facility_enriched.pkl")
covenant = pd.read_pickle(OUTPUT_DIR/"02_covenant_enriched.pkl")
asset = pd.read_pickle(OUTPUT_DIR/"01_clean_asset.pkl")
coverage = pd.read_csv(OUTPUT_DIR/"coverage.csv")
events = pd.read_pickle(OUTPUT_DIR/"events.pkl")


In [ ]:
out = OUTPUT_DIR/"structured_data_cleaned.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as writer:
    rmpm.to_excel(writer,sheet_name="RMPM",index=False)
    legal_entity.to_excel(writer,sheet_name="Legal_Entity",index=False)
    facility.to_excel(writer,sheet_name="Facility",index=False)
    covenant.to_excel(writer,sheet_name="Covenant",index=False)
    asset.to_excel(writer,sheet_name="Asset",index=False)
    coverage.to_excel(writer,sheet_name="Coverage",index=False)
    events.to_excel(writer,sheet_name="Events",index=False)
print(out)


## 10. Compléments EDA pour le rapport PFE

**Objectif :** produire des statistiques vérifiables et des figures prêtes à intégrer au rapport, sans créer de résultats fictifs.

**Attention :** les données utilisées restent sur le poste BNP et les sorties sont à conserver dans un emplacement autorisé. Les figures remplacent les identifiants par `CP_01`, `CP_02`, etc. Une anonymisation des libellés seule ne garantit pas l'absence d'informations sensibles.

**Exécution :** exécuter les cellules précédentes, puis celles de cette section. Les résultats sont produits à partir des `.pkl` déjà créés par le notebook. Les 3 cellules d'origine corrigées concernent `PD_CLEAN`, `IS_BREACH` et la constitution des événements Covenant.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

FIG_DIR = OUTPUT_DIR / "figures_pfe"
STATS_DIR = OUTPUT_DIR / "statistiques_pfe"
FIG_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = ["rmpm", "legal_entity", "facility", "covenant", "asset"]
NAMES = {"rmpm": "RMPM", "legal_entity": "Legal Entity", "facility": "Facility",
         "covenant": "Covenant", "asset": "Asset"}
RAW = {s: pd.read_pickle(OUTPUT_DIR / f"00_raw_{s}.pkl") for s in SOURCES}
CLEAN = {s: pd.read_pickle(OUTPUT_DIR / f"01_clean_{s}.pkl") for s in SOURCES}

# Les alias n'apparaissent QUE dans les figures et tableaux agreges.
CP_IDS = sorted({str(v).strip() for v in TARGET_RMPM_IDS})
CP_ALIAS = {cp: f"CP_{i:02d}" for i, cp in enumerate(CP_IDS, start=1)}

def col_key(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

def pick_col(df, *names):
    actual = {col_key(c): c for c in df.columns}
    for candidate in names:
        if col_key(candidate) in actual:
            return actual[col_key(candidate)]
    return None

def parse_dates(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_localize(None)

def numeric_values(s):
    # Suffisant pour les ratios/indicateurs simples. Revoir les unites au besoin.
    def one(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
        value = str(x).strip().replace("\u00a0", "").replace("\u202f", "").replace(" ", "")
        value = value.replace("%", "")
        if "," in value and "." in value:
            if value.rfind(",") > value.rfind("."):
                value = value.replace(".", "").replace(",", ".")
            else:
                value = value.replace(",", "")
        elif "," in value:
            value = value.replace(",", ".")
        try:
            return float(value)
        except (TypeError, ValueError):
            return np.nan
    return s.map(one).astype(float)

def save_figure(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    print("Figure enregistree :", name)
    plt.show()

print("Fichiers charges :", {NAMES[s]: CLEAN[s].shape for s in SOURCES})
print("Repertoire figures :", FIG_DIR)
print("Repertoire statistiques :", STATS_DIR)

### 10.1 – Tableau descriptif et périmètre temporel

Utiliser ce tableau dans la **présentation des datasets**. `brut` et `nettoyé` ne couvrent pas forcément le même périmètre : le nettoyage comprend le filtrage des contreparties retenues. Ne pas interpréter mécaniquement l'écart entre leurs dimensions comme une simple suppression de données invalides.

In [ ]:
DATE_FIELDS = {
    "rmpm": ["SITUATION_DATE", "DATE_OF_SITUATION"],
    "legal_entity": ["DATE_OF_COMMITTEE", "REQUEST_CREATION_DATE"],
    "facility": ["DATE_OF_COMMITTEE", "CREATION_DATE", "MATURITY_DATE"],
    "covenant": ["REPORTING_DATE", "PERFORMED_ON", "RECEIPT_DATE", "DUE_DATE"],
    "asset": ["VALUATION_DATE", "ASSET_VALUATION_DATE", "DATE_OF_COMMITTEE"]
}

rows = []
for src in SOURCES:
    for stage, frame in [("Brut", RAW[src]), ("Apres preparation + filtre", CLEAN[src])]:
        id_col = pick_col(frame, "RMPM_ID")
        date_col = pick_col(frame, *DATE_FIELDS[src])
        dates = parse_dates(frame[date_col]) if date_col else pd.Series(dtype="datetime64[ns]")
        rows.append({
            "Source": NAMES[src], "Perimetre": stage,
            "Lignes": len(frame), "Colonnes": len(frame.columns),
            "Contreparties": frame[id_col].nunique(dropna=True) if id_col else np.nan,
            "Doublons exacts": int(frame.duplicated().sum()),
            "Periode debut": dates.min().date().isoformat() if dates.notna().any() else "",
            "Periode fin": dates.max().date().isoformat() if dates.notna().any() else "",
            "Colonne temporelle": date_col or "absente"
        })
profile_df = pd.DataFrame(rows)
display(profile_df)
profile_df.to_csv(STATS_DIR / "01_description_datasets.csv", index=False)

print("NB : les dates ci-dessus decrivent uniquement la couverture de chaque extrait, pas une frequence de suivi prouvee.")

### 10.1 bis – Schéma de relations entre les datasets

**Schéma logique simplifié** (à placer dans la description des sources). Les flèches indiquent les identifiants permettant d'envisager des rapprochements. Elles ne prouvent pas qu'une jointure sera toujours possible ou univoque ; contrôler les cardinalités dans les données.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(figsize=(11.0, 6.0))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
boxes = {
    "RMPM": (0.04, .55, .22, .25),
    "Legal Entity": (.39, .72, .23, .19),
    "Facility": (.39, .35, .23, .20),
    "Covenant": (.75, .59, .22, .20),
    "Asset": (.75, .24, .22, .20),
}
notes = {
    "RMPM": "PD · Rating · GRR\nSituations datees",
    "Legal Entity": "Decisions · Watchlist\nUTP · NPE",
    "Facility": "Engagements · FBE\nMontants · Echeances",
    "Covenant": "Seuils · Statuts\nBREACHED",
    "Asset": "Garanties · Actifs\nValorisations"
}
for name, (x, y, w, h) in boxes.items():
    ax.add_patch(FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.013,rounding_size=0.025", facecolor="#f5f7fa",
        edgecolor="#435b75", linewidth=1.4))
    ax.text(x + w/2, y + h*.70, name, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(x + w/2, y + h*.31, notes[name], ha="center", va="center", fontsize=9)

def arrow(p1, p2, label, shift=(0,0)):
    ax.add_patch(FancyArrowPatch(p1,p2,arrowstyle="-|>", mutation_scale=13,
                                 color="#5e7692",linewidth=1.6,connectionstyle="arc3"))
    ax.text((p1[0]+p2[0])/2+shift[0],(p1[1]+p2[1])/2+shift[1],
            label, fontsize=9, ha="center", va="center", color="#33475b",
            bbox=dict(facecolor="white", edgecolor="none", pad=1.2))
arrow((.26,.73),(.39,.80),"RMPM_ID",(0,.045))
arrow((.26,.60),(.39,.48),"RMPM_ID",(0,-.045))
arrow((.62,.50),(.75,.68),"FACILITY_UNIQUE_ID",(0,.03))
arrow((.62,.41),(.75,.34),"FACILITY_UNIQUE_ID",(0,-.04))
ax.set_title("Sources structurees : relations logiques (schema simplifie)", fontsize=13, pad=17)
fig.tight_layout()
save_figure("F00_schema_sources")
print("Les jointures doivent etre verifiees au niveau du couple contrepartie + identifiant facility, lorsque present.")

### 10.2 – Valeurs manquantes et qualité

Deux niveaux : **taux global** par fichier, puis **taux par variable métier**. Une variable absente du fichier est indiquée séparément d'une colonne présente mais entièrement vide. Certaines absences (date d'entrée en Watchlist, par exemple) peuvent signifier « non applicable » et pas « mauvaise qualité ».

La figure globale est une aide descriptive ; le taux de complétude des variables essentielles est plus informatif pour conclure sur l'aptitude à détecter des EWS.

In [ ]:
# Tableau de completude : valeurs nulles selon pandas, avant/apres.
missing_rows = []
for src in SOURCES:
    for stage, df in [("Brut", RAW[src]), ("Apres preparation + filtre", CLEAN[src])]:
        cells = df.shape[0] * df.shape[1]
        miss = int(df.isna().sum().sum())
        missing_rows.append({
            "Source": NAMES[src], "Perimetre": stage, "Cellules": cells,
            "Cellules manquantes": miss,
            "Taux manquant (%)": round(100 * miss / cells, 2) if cells else np.nan,
            "Colonnes totalement vides": int(df.isna().all().sum()),
            "Doublons exacts": int(df.duplicated().sum())
        })
missing_global = pd.DataFrame(missing_rows)
display(missing_global)
missing_global.to_csv(STATS_DIR / "02_missing_global.csv", index=False)

raw_missing = (missing_global[missing_global["Perimetre"] == "Brut"]
               .set_index("Source")["Taux manquant (%)"].sort_values())
fig, ax = plt.subplots(figsize=(9, 4.3))
bars = ax.barh(raw_missing.index, raw_missing.values)
for rect, value in zip(bars, raw_missing.values):
    ax.text(min(value + 1, 99), rect.get_y() + rect.get_height()/2,
            f"{value:.1f} %", va="center", fontsize=10)
ax.set_xlim(0, 105)
ax.set_xlabel("Cellules manquantes (%)")
ax.set_title("Qualite des sources : taux de valeurs manquantes (donnees brutes)")
ax.grid(axis="x", alpha=.2)
fig.tight_layout()
save_figure("F01_valeurs_manquantes_globales")

# Variables metier prioritaires : adapter si les noms different dans les sources BNP.
ESSENTIAL = {
    "rmpm": ["RMPM_ID", "SITUATION_DATE", "PD_CLEAN", "GRR_SU", "COUNTERPARTY_RATING", "RATING_NUM"],
    "legal_entity": ["RMPM_ID", "DATE_OF_COMMITTEE", "COUNTERPARTY_RATING", "WATCHLIST_INDICATOR",
                     "UTP_TEST", "CONFIRMED_UTP", "NPE", "CLOSE_TO_DEFAULT"],
    "facility": ["RMPM_ID", "FACILITY_UNIQUE_ID", "DATE_OF_COMMITTEE", "FINAL_TAKE_AMOUNT",
                 "OUTSTANDING_GLOBAL", "MATURITY_DATE", "IS_FBE"],
    "covenant": ["RMPM_ID", "FACILITY_UNIQUE_ID", "REPORTING_DATE", "PERFORMED_ON",
                 "BREACH_STATUS", "CURRENT_VALUE", "THRESHOLD_VALUE"],
    "asset": ["RMPM_ID", "FACILITY_UNIQUE_ID", "VALUATION_DATE", "VALUATION_AMOUNT", "ASSET_STATUS"]
}

field_rows = []
for src in SOURCES:
    for requested in ESSENTIAL[src]:
        for stage, df in [("Brut", RAW[src]), ("Apres preparation + filtre", CLEAN[src])]:
            col = pick_col(df, requested)
            field_rows.append({
                "Source": NAMES[src], "Variable attendue": requested,
                "Colonne retrouvee": col or "ABSENTE", "Perimetre": stage,
                "Taux manquant (%)": round(100 * df[col].isna().mean(), 2) if col and len(df) else np.nan,
                "Valeurs non nulles": int(df[col].notna().sum()) if col else 0,
                "Variable disponible": bool(col)
            })
missing_fields = pd.DataFrame(field_rows)
display(missing_fields[missing_fields["Perimetre"] == "Apres preparation + filtre"])
missing_fields.to_csv(STATS_DIR / "03_missing_variables_metier.csv", index=False)

selected = (missing_fields[(missing_fields["Perimetre"] == "Apres preparation + filtre")
                           & missing_fields["Variable disponible"]]
            .dropna(subset=["Taux manquant (%)"]).copy())
selected["Label"] = selected["Source"] + " · " + selected["Variable attendue"]
selected = selected[selected["Taux manquant (%)"] > 0]
selected = selected.sort_values("Taux manquant (%)", ascending=False).head(16)
if not selected.empty:
    fig, ax = plt.subplots(figsize=(11, max(5.5, .42*len(selected))))
    ax.barh(selected["Label"][::-1], selected["Taux manquant (%)"][::-1])
    ax.set_xlim(0, 105)
    ax.set_xlabel("Valeurs manquantes (%)")
    ax.set_title("Variables metier les plus incompletes (donnees preparees)")
    ax.grid(axis="x", alpha=.2)
    fig.tight_layout()
    save_figure("F01b_valeurs_manquantes_variables")
else:
    print("Aucune valeur manquante sur les variables metier disponibles pour cette analyse.")

### 10.2 bis – Contrôles des identifiants, dates et anomalies

Contrôles complémentaires qui renforcent la méthodologie EDA. Les clés ci-dessous sont des **clés candidates** : un doublon sur ces colonnes peut être légitime selon la granularité, et nécessite une vérification métier.

In [ ]:
CANDIDATE_KEYS = {
    "rmpm": ["RMPM_ID", "SITUATION_DATE"],
    "legal_entity": ["RMPM_ID", "DATE_OF_COMMITTEE"],
    "facility": ["RMPM_ID", "FACILITY_UNIQUE_ID", "DATE_OF_COMMITTEE"],
    "covenant": ["RMPM_ID", "FACILITY_UNIQUE_ID", "COVENANT_SUR_NAME", "REPORTING_DATE"],
    "asset": ["RMPM_ID", "FACILITY_UNIQUE_ID", "ASSET_FUNCTIONAL_ID"]
}
quality_rows=[]
for src in SOURCES:
    frame=CLEAN[src]
    id_col=pick_col(frame, "RMPM_ID")
    date_col=pick_col(frame, *DATE_FIELDS[src])
    keys=[pick_col(frame,k) for k in CANDIDATE_KEYS[src]]
    keys=[k for k in keys if k]
    quality_rows.append({
        "Source": NAMES[src],
        "Lignes": len(frame),
        "Identifiants contrepartie manquants": int(frame[id_col].isna().sum()) if id_col else np.nan,
        "Dates manquantes (date choisie)": int(parse_dates(frame[date_col]).isna().sum()) if date_col else np.nan,
        "Doublons exacts": int(frame.duplicated().sum()),
        "Doublons de cle candidate": int(frame.duplicated(subset=keys).sum()) if keys else np.nan,
        "Cles testees": ", ".join(keys) if keys else "Aucune cle disponible"
    })
quality_df=pd.DataFrame(quality_rows)
display(quality_df)
quality_df.to_csv(STATS_DIR / "07_controles_qualite.csv", index=False)

rmpm_raw=RAW["rmpm"]
original_pd_col=pick_col(rmpm_raw,"PD_CLEAN")
if original_pd_col:
    raw_pd=numeric_values(rmpm_raw[original_pd_col])
    print("PD = 999.99 dans le brut :", int(raw_pd.eq(999.99).sum()))
    print("PD negatives dans le brut :", int(raw_pd.lt(0).sum()))
    print("PD > 100 dans le brut :", int(raw_pd.gt(100).sum()))
    print("Verifier l'unite de PD (fraction ou pourcentage) avant toute interpretation.")
print("Les controles portent sur les sources disponibles, sans validation metier des anomalies.")

### 10.3 – Statistiques et couverture des événements effectivement détectés

La table `events.pkl` contient des **changements de valeurs et des statuts observés**, pas des EWS validés ni des prédictions. Les événements de Facility ne sont pas automatiquement des alertes de risque. Les observations `Covenant breach observed` indiquent des constats, pas nécessairement des incidents distincts.

In [ ]:
EVENT_DOMAINS = [
    "Credit Monitoring", "Covenants et FBE", "Endettement et refinancement",
    "Performance financiere et liquidite", "Gouvernance", "ESG", "Autres evenements"
]
EVALUATED = {"Credit Monitoring", "Covenants et FBE", "Endettement et refinancement"}

def event_domain(row):
    src = str(row.get("SOURCE", "")).upper()
    ev = str(row.get("EVENT_TYPE", "")).upper()
    if src in {"RMPM", "LEGAL", "LEGAL_ENTITY"}:
        return "Credit Monitoring"
    if src == "COVENANT" or (src == "FACILITY" and "FBE" in ev):
        return "Covenants et FBE"
    if src == "FACILITY":
        return "Endettement et refinancement"
    return "Autres evenements"

events_report = pd.read_pickle(OUTPUT_DIR / "events.pkl").copy()
if events_report.empty:
    events_report = pd.DataFrame(columns=["RMPM_ID", "DATE", "SOURCE", "EVENT_TYPE"])

# Garde-fou si cette cellule est lancee avec une ancienne version du pipeline :
# le texte 'NOT BREACHED' ne doit pas compter comme une rupture.
is_cov = events_report.get("SOURCE", pd.Series(dtype="object")).astype(str).str.upper().eq("COVENANT")
labels = events_report.get("EVENT_TYPE", pd.Series(dtype="object")).astype(str).str.lower()
is_negative = labels.str.contains(r"\b(not|no|non|without)\s+breach(ed)?\b|unbreached", regex=True, na=False)
is_informative = labels.str.contains(r"breach|waiv", regex=True, na=False)
removed = int((is_cov & (is_negative | ~is_informative)).sum())
events_report = events_report.loc[~(is_cov & (is_negative | ~is_informative))].copy()
if removed:
    print(f"AVERTISSEMENT : {removed} faux evenements de Covenant retires de l'analyse PFE. Relancer les cellules 31 a 42 corrigees.")

events_report["RMPM_ID"] = events_report["RMPM_ID"].astype("string").str.strip()
events_report["DATE"] = parse_dates(events_report["DATE"])
events_report = events_report.dropna(subset=["RMPM_ID", "DATE"])
events_report["CONTREPARTIE"] = events_report["RMPM_ID"].map(CP_ALIAS).fillna("CP_HORS_PERIMETRE")
events_report["DOMAINE"] = events_report.apply(event_domain, axis=1)

detail_events = (events_report.groupby(["DOMAINE", "SOURCE", "EVENT_TYPE"], dropna=False)
                 .agg(Observations=("DATE", "size"), Contreparties=("CONTREPARTIE", "nunique"))
                 .reset_index().sort_values("Observations", ascending=False))
display(detail_events)
detail_events.to_csv(STATS_DIR / "04_evenements_detectes_par_type.csv", index=False)

domain_rows = []
for domain in EVENT_DOMAINS:
    subset = events_report[events_report["DOMAINE"] == domain]
    measured = domain in EVALUATED
    domain_rows.append({
        "Domaine": domain,
        "Regles de detection": "Implementees (partiellement)" if measured else "Non implementees",
        "Observations evenements": len(subset) if measured else pd.NA,
        "Contreparties avec evenement": subset["CONTREPARTIE"].nunique() if measured else pd.NA
    })
domain_stats = pd.DataFrame(domain_rows)
display(domain_stats)
domain_stats.to_csv(STATS_DIR / "05_couverture_domaines_ews.csv", index=False)

if not detail_events.empty:
    tops = (events_report["EVENT_TYPE"].value_counts().head(12).sort_values())
    fig, ax = plt.subplots(figsize=(11, max(5, 0.44 * len(tops))))
    ax.barh(tops.index, tops.values)
    ax.set_xlabel("Nombre d'observations d'evenements")
    ax.set_title("Changements et evenements detectes dans les donnees structurees")
    ax.grid(axis="x", alpha=.2)
    fig.tight_layout()
    save_figure("F04a_types_evenements_detectes")
else:
    print("Aucun evenement retenu par le pipeline sur ce perimetre.")

### 10.3 bis – Qualification des observations

Ce tableau distingue les **ruptures de covenants constatées** des simples variations de mesure, des changements de statut et des mises à jour d'engagements. Il est préférable à un décompte brut d'« EWS trouvés ». La qualification est indicative et ne remplace pas la validation métier.

In [ ]:
def observation_nature(row):
    src=str(row.get("SOURCE","")).upper()
    label=str(row.get("EVENT_TYPE","")).lower()
    if src=="COVENANT" and "breach" in label:
        return "Rupture contractuelle constatee"
    if src=="COVENANT" and "waiv" in label:
        return "Waiver constate"
    if src in {"LEGAL", "LEGAL_ENTITY"}:
        return "Changement de statut de risque (constat)"
    if src=="RMPM":
        return "Variation d'indicateur (non qualifiee EWS)"
    if src=="FACILITY":
        return "Modification d'engagement (a contextualiser)"
    return "Autre observation (a examiner)"

events_report["NATURE_OBSERVATION"]=events_report.apply(observation_nature, axis=1)
nature_stats=(events_report.groupby("NATURE_OBSERVATION",dropna=False)
              .agg(Observations=("DATE","size"), Contreparties=("CONTREPARTIE","nunique"))
              .reset_index().sort_values("Observations",ascending=False))
display(nature_stats)
nature_stats.to_csv(STATS_DIR / "08_qualification_observations.csv", index=False)
print("Remarque : les changements ne sont pas des signaux precoces confirmes sans validation temporelle et metier.")

### 10.4 – Matrice de couverture des événements par contrepartie

Dans la matrice : `0` = règles en place mais aucun événement détecté ; `NE` = **non évalué**, et non « absence démontrée d'EWS ». Un nombre positif correspond au **nombre d'observations d'événements**, pas au nombre d'alertes précoces validées.

In [ ]:
cp_names = [CP_ALIAS[cp] for cp in CP_IDS]
coverage_matrix = pd.DataFrame(np.nan, index=EVENT_DOMAINS, columns=cp_names, dtype=float)
for domain in EVALUATED:
    coverage_matrix.loc[domain] = 0.0
for (domain, cp), count in events_report.groupby(["DOMAINE", "CONTREPARTIE"]).size().items():
    if domain in EVALUATED and cp in coverage_matrix.columns:
        coverage_matrix.loc[domain, cp] = float(count)

print("Matrice (NaN = non evalue) :")
display(coverage_matrix)
coverage_matrix.to_csv(STATS_DIR / "06_matrice_couverture_evenements.csv", index_label="Domaine")

fig, ax = plt.subplots(figsize=(10.5, 5.1))
values = np.ma.masked_invalid(coverage_matrix.to_numpy(dtype=float))
cmap = plt.cm.Blues.copy()
cmap.set_bad("#e8e8e8")
im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=0,
               vmax=max(1, float(np.nanmax(coverage_matrix.to_numpy())) if np.isfinite(coverage_matrix.to_numpy()).any() else 1))
ax.set_xticks(range(len(cp_names)), cp_names)
ax.set_yticks(range(len(EVENT_DOMAINS)), EVENT_DOMAINS)
for i in range(len(EVENT_DOMAINS)):
    for j in range(len(cp_names)):
        x = coverage_matrix.iat[i,j]
        ax.text(j, i, "NE" if np.isnan(x) else str(int(x)), ha="center", va="center",
                color="white" if np.isfinite(x) and x > im.norm.vmax * .55 else "black", fontsize=10)
ax.set_title("Couverture des evenements detectes par domaine et contrepartie")
fig.colorbar(im, ax=ax, label="Observations evenementielles (pas EWS valides)", shrink=.77)
fig.tight_layout()
save_figure("F04b_matrice_couverture_ews")

print("NE = domaine non evalue par les regles de detection du notebook.")
print("Attention : 5 contreparties ne suffisent pas a mesurer une capacite predictive.")

### 10.5 – Figure PD + rating à conserver dans le rapport

La PD est tracée **dans son unité native** : il faut confirmer auprès de l'équipe si la source contient des fractions (0 à 1) ou des pourcentages (0 à 100). Le `RATING_NUM` est un encodage numérique technique dont le **sens de l'échelle** doit être vérifié avant de parler de dégradation. Une courbe seule ne démontre pas l'anticipation d'un risque.

La cellule choisit automatiquement une contrepartie avec suffisamment de dates. Pour en imposer une : saisir un identifiant dans `CP_RMPM = "..."`, uniquement dans l'environnement autorisé.

In [ ]:
CP_RMPM = None  # Exemple : "identifiant reel dans l'environnement BNP" ; ne pas publier
rmpm_fig = pd.read_pickle(OUTPUT_DIR / "02_rmpm_enriched.pkl").copy()
date_col = pick_col(rmpm_fig, "SITUATION_DATE")
pd_col = pick_col(rmpm_fig, "PD_CLEAN")
rating_col = pick_col(rmpm_fig, "RATING_NUM")

if date_col and (pd_col or rating_col):
    rmpm_fig["__DATE__"] = parse_dates(rmpm_fig[date_col])
    if pd_col:
        rmpm_fig["__PD__"] = numeric_values(rmpm_fig[pd_col]).mask(
            lambda x: x.eq(999.99) | x.lt(0) | x.gt(100))
    if rating_col:
        rmpm_fig["__RATING__"] = numeric_values(rmpm_fig[rating_col])

    ordered = sorted(CP_IDS,
        key=lambda cp: len(rmpm_fig.loc[rmpm_fig["RMPM_ID"].astype(str).eq(cp) & rmpm_fig["__DATE__"].notna()]),
        reverse=True)
    chosen = str(CP_RMPM) if CP_RMPM else (ordered[0] if ordered else None)
    plot_data = (rmpm_fig.loc[rmpm_fig["RMPM_ID"].astype(str).eq(chosen)]
                 .dropna(subset=["__DATE__"]).sort_values("__DATE__")) if chosen else pd.DataFrame()
    if plot_data.empty:
        print("Aucune serie temporelle exploitable pour la contrepartie choisie.")
    else:
        panels = [("__PD__", "PD (unite source a verifier)")] if pd_col else []
        if rating_col:
            panels.append(("__RATING__", "Rating (encodage numerique)"))
        fig, axes = plt.subplots(len(panels), 1, figsize=(10.5, 6.0), sharex=True, squeeze=False)
        for ax, (measure, label) in zip(axes[:, 0], panels):
            ax.plot(plot_data["__DATE__"], plot_data[measure], marker="o", linewidth=1.6)
            ax.set_ylabel(label)
            ax.grid(alpha=.25)
        axes[-1, 0].set_xlabel("Date de situation")
        fig.suptitle(f"Evolution du risque de credit - {CP_ALIAS.get(chosen, 'CP')}" )
        fig.autofmt_xdate()
        fig.tight_layout()
        save_figure("F02_evolution_pd_rating")
        print("Contrepartie figure :", CP_ALIAS.get(chosen, "hors perimetre"))
else:
    print("Colonnes date / PD / rating manquantes : verifier les colonnes RMPM.")

### 10.6 – Figure Covenant : valeur, seuil et ruptures constatées

Cette figure suppose que `CURRENT_VALUE` et la colonne de seuil sont comparables et qu'on représente **un seul covenant homogène** (même ratio / unité / facility). L'orientation du seuil (`≤` ou `≥`) n'est **pas** inférée automatiquement. `BREACHED` est traité comme un constat. Plusieurs observations `BREACHED` ne sont pas forcément plusieurs ruptures distinctes.

Si aucune observation complète n'est disponible, la cellule signale le blocage au lieu d'inventer les valeurs du PDF.

In [ ]:
CP_COVENANT = None  # Option : selectionner une contrepartie dans l'environnement BNP
cov_fig = pd.read_pickle(OUTPUT_DIR / "02_covenant_enriched.pkl").copy()
cov_date = pick_col(cov_fig, "COVENANT_EVENT_DATE", "PERFORMED_ON", "REPORTING_DATE", "RECEIPT_DATE", "DUE_DATE")
cov_current = pick_col(cov_fig, "CURRENT_VALUE", "ACTUAL_VALUE")
cov_threshold = pick_col(cov_fig, "THRESHOLD_VALUE", "MAX_VALUE", "MIN_VALUE")
cov_type = pick_col(cov_fig, "COVENANT_SUR_NAME", "COVENANT_TYPE", "TYPE")
cov_fac = pick_col(cov_fig, "FACILITY_UNIQUE_ID")
cov_status = pick_col(cov_fig, "IS_BREACH", "BREACH_STATUS", "BREACHED_STATUS")

if not cov_date or not cov_current:
    print("Impossible de tracer : date et/ou CURRENT_VALUE absentes.")
else:
    cov_fig["__DATE__"] = parse_dates(cov_fig[cov_date])
    cov_fig["__VALUE__"] = numeric_values(cov_fig[cov_current])
    if cov_threshold:
        cov_fig["__THRESHOLD__"] = numeric_values(cov_fig[cov_threshold])
    if cov_status == "IS_BREACH":
        cov_fig["__BREACH__"] = cov_fig[cov_status].fillna(False).astype(bool)
    elif cov_status:
        cov_fig["__BREACH__"] = cov_fig[cov_status].map(parse_breach_status).astype(bool)
    else:
        cov_fig["__BREACH__"] = False

    group_keys = ["RMPM_ID"]
    if cov_fac: group_keys.append(cov_fac)
    if cov_type: group_keys.append(cov_type)
    candidates = []
    for key, group in cov_fig.groupby(group_keys, dropna=False):
        cp = str(group["RMPM_ID"].iloc[0])
        if CP_COVENANT is not None and cp != str(CP_COVENANT):
            continue
        usable = group.dropna(subset=["__DATE__", "__VALUE__"])
        if len(usable):
            candidates.append((int(usable["__BREACH__"].sum()), len(usable), key, usable))
    candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    if not candidates:
        print("Pas de covenant exploitable avec date et valeur numerique dans ce perimetre.")
    else:
        _, _, chosen_key, plot_cov = candidates[0]
        plot_cov = plot_cov.sort_values("__DATE__")
        cp = str(plot_cov["RMPM_ID"].iloc[0])
        label = str(plot_cov[cov_type].iloc[0]) if cov_type else "Covenant financier"
        fig, ax = plt.subplots(figsize=(10.5, 5.0))
        ax.plot(plot_cov["__DATE__"], plot_cov["__VALUE__"], marker="o",
                linewidth=1.5, label="Valeur observee")
        if cov_threshold and plot_cov["__THRESHOLD__"].notna().any():
            ax.plot(plot_cov["__DATE__"], plot_cov["__THRESHOLD__"], "--",
                    linewidth=1.6, label="Seuil contractuel")
        breach_obs = plot_cov.loc[plot_cov["__BREACH__"]]
        if len(breach_obs):
            ax.scatter(breach_obs["__DATE__"], breach_obs["__VALUE__"],
                       s=115, marker="x", linewidths=2.2, label="Rupture constatee", zorder=5)
        ax.set_xlabel("Date de reporting / de constat")
        ax.set_ylabel("Valeur / seuil (unites a verifier)")
        ax.set_title(f"Covenant : {label} - {CP_ALIAS.get(cp, 'CP')}")
        ax.grid(alpha=.25)
        ax.legend()
        fig.autofmt_xdate()
        fig.tight_layout()
        save_figure("F03_covenant_seuil_breach")
        print("Observations retenues :", len(plot_cov), "| dont BREACHED :", len(breach_obs))
        print("Verification avant publication : sens du seuil, unite, date et identite du covenant.")

### 10.7 – Contrôles complémentaires, interprétation et exports

Les tableaux et les figures sont stockés dans `output/statistiques_pfe/` et `output/figures_pfe/` (ou à l'emplacement défini dans `config.py`).

- **Pour la présentation des sources** : `01_description_datasets.csv`, figure `F00*`.
- **Pour la qualité des données** : `02_missing_global.csv`, `03_missing_variables_metier.csv`, `07_controles_qualite.csv`, figures `F01*`.
- **Pour les observations détectées** : `04_evenements_detectes_par_type.csv`, `05_couverture_domaines_ews.csv`, `06_matrice_couverture_evenements.csv`, `08_qualification_observations.csv`, figures `F04*`.
- **Pour les évolutions temporelles** : figures `F02*` et `F03*` si les observations requises sont présentes.

**Formulation recommandée :** « Les données structurées permettent d'identifier des changements d'indicateurs et des ruptures contractuelles constatées, mais ne suffisent pas à documenter de façon homogène les signaux précoces qualitatifs recherchés. »

In [ ]:
print("BILAN DES SORTIES POUR LE RAPPORT")
print("- Contreparties dans le perimetre :", len(CP_IDS))
print("- Observations evenementielles retenues :", len(events_report))
print("- Types d'evenements distincts :", events_report["EVENT_TYPE"].nunique() if not events_report.empty else 0)
print("- Contreparties avec evenements detectes :", events_report["CONTREPARTIE"].nunique() if not events_report.empty else 0)
print("- Observations Covenant breach :", int((events_report["EVENT_TYPE"].astype(str).str.lower()
                                  .str.contains("covenant breach observed", regex=False)).sum()))
print("- Repartition des domaines :")
display(domain_stats)

print("\nFigures generees :")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" ", p.name)
print("\nTableaux generes :")
for p in sorted(STATS_DIR.glob("*.csv")):
    print(" ", p.name)

print("\nIMPORTANT : NE / Non implemente = information non evaluee, pas absence prouvee de risque.")
print("Aucune conclusion predictive ne doit etre tiree des seuls comptages.")